<a href="https://colab.research.google.com/github/Arda-Avci/AI-Publisher/blob/main/Google_Colab_AI_Publisher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 AI Publisher - Otonom Video Üretim ve Pazarlama Sunucusu

Bu notebook, **AI Publisher** projesi için Google Colab üzerinde çalışan GPU destekli yapay zekâ sunucusunu (CogVideoX-5b, XTTS-v2, AudioLDM2, GFPGAN/RealESRGAN ve Wav2Lip dudak senkronizasyonu) kurar ve başlatır.

### 💡 Kurulum ve Başlatma Talimatı:
1. Üst menüden **Düzenle (Edit) > Defter Ayarları (Notebook Settings)** seçeneğine gidin.
2. Donanım ivmelendiriciyi **T4 GPU** (veya daha üstü bir ekran kartı) olarak seçip kaydedin.
3. Aşağıdaki kod hücresini çalıştırın. Bağımlılıklar ilk kez kurulurken kernel otomatik olarak yeniden başlatılacaktır (Oturum çöktü uyarısı normaldir).
4. Kernel otomatik kapandıktan sonra **aynı hücreyi ikinci kez çalıştırın**.
5. Ngrok Auth Token istendiğinde ngrok.com adresinden alacağınız token'ınızı girin.
6. Sunucu başarıyla ayağa kalktığında size verilen tünel URL'sini kopyalayıp yerel Node.js projenizin `.env` dosyasındaki `COLAB_URL` alanına yapıştırın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GITHUB_TOKEN = "" #@param {type:"string"}
NGROK_TOKEN = "" #@param {type:"string"}

import os, subprocess, shutil, sys
from google.colab import userdata

repo_dir = "/content/AI-Publisher"

if os.path.exists(repo_dir):
    print("Deleting existing repository directory...")
    shutil.rmtree(repo_dir, ignore_errors=True)

if not GITHUB_TOKEN:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    GITHUB_TOKEN = input("GitHub token: ").strip()

if not NGROK_TOKEN:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
if not NGROK_TOKEN:
    NGROK_TOKEN = input("Ngrok Auth Token: ").strip()

repo_url = f"https://{GITHUB_TOKEN}@github.com/Arda-Avci/AI-Publisher.git"

print("Repository klonlanıyor...")
subprocess.run(["git", "clone", repo_url, repo_dir], check=True)

print("colab_setup.py kopyalanıyor...")
shutil.copy(os.path.join(repo_dir, "colab_setup.py"), "/content/colab_setup.py")
print("colab_server.py kopyalanıyor...")
shutil.copy(os.path.join(repo_dir, "colab_server.py"), "/content/colab_server.py")

print("Kurulum betiği canlı loglar eşliğinde çalıştırılıyor...")
try:
    env = os.environ.copy()
    env["GITHUB_TOKEN"] = GITHUB_TOKEN
    env["NGROK_TOKEN"] = NGROK_TOKEN
    
    process = subprocess.Popen(
        ["python3", "-u", "/content/colab_setup.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    # Canlı logları yazdır
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    process.wait()
    
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, process.args)
except subprocess.CalledProcessError as e:
    if e.returncode in [9, -9, 137, 100]:
        print(f"\n[INFO] Kurulum betiği sonlandı (Çıkış Kodu: {e.returncode}). Oturum yenileniyor...")
        import os
        os.kill(os.getpid(), 9)
    else:
        print(f"\n❌ Kurulum hatayla bitti! Çıkış Kodu: {e.returncode}")
        raise e

## 🛠️ Seçenek C: Tüm Docker İmajlarını İnşa Et ve Google Drive'a Kaydet (Maliyet Tasarruflu CPU Modu)

Bu bölüm, AI Publisher projesinde kullanılan 11 Docker konteyner imajını Google Colab **CPU** çalışma zamanında sıfırdan inşa eder, `pigz` kullanarak çok çekirdekli paralel sıkıştırır, Google Drive'a yükler ve bütünlük kontrolüyle doğrular.

### 💡 Maliyet Tasarrufu ve Çalıştırma Talimatı:
1. Üst menüden **Düzenle (Edit) > Defter Ayarları (Notebook Settings)** seçeneğine gidin.
2. Donanım ivmelendiriciyi **CPU** (GPU yok) olarak seçip kaydedin (CPU modu ücretsizdir veya çok daha ucuzdur).
3. Aşağıdaki hücreyi çalıştırın. Hücre, tüm imajları sırayla derleyecek, Drive'a `.tar.gz` olarak kaydedecek ve bütünlüklerini doğrulayacaktır.
4. **Otomatik Kapanma (Unassign):** Derleme ve doğrulama tamamlandığında, `AUTO_DISCONNECT = True` olarak ayarlandıysa, Colab oturumu otomatik olarak sonlandırılacaktır. Bu sayede işlem bittiğinde ek süre için ücretlendirilmezsiniz.

In [ ]:
#@title 🛠️ Seçenek C: Tüm Docker İmajlarını İnşa Et (Maliyet Tasarruflu CPU Modu)
GITHUB_TOKEN = "" #@param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')

import subprocess
import os
import time
import urllib.request
import sys
import shutil
from google.colab import userdata

# 1. Depoyu Klonla (Eğer yoksa)
repo_dir = "/content/AI-Publisher"
if not os.path.exists(repo_dir):
  print("Repository klonlanıyor...")
  if not GITHUB_TOKEN:
    try:
      GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except: pass
  if not GITHUB_TOKEN:
    GITHUB_TOKEN = input("GitHub token: ").strip()
  
  repo_url = f"https://{GITHUB_TOKEN}@github.com/Arda-Avci/AI-Publisher.git"
  subprocess.run(["git", "clone", repo_url, repo_dir], check=True)

# 2. Go Registry İndir ve Kur
if not os.path.exists("/usr/local/bin/registry"):
  print("[INFO] Docker Registry binary indiriliyor...")
  subprocess.run("wget -q https://github.com/distribution/distribution/releases/download/v2.8.2/registry_2.8.2_linux_amd64.tar.gz", shell=True)
  subprocess.run("tar -xzf registry_2.8.2_linux_amd64.tar.gz registry", shell=True)
  subprocess.run("mv registry /usr/local/bin/registry", shell=True)
  subprocess.run("rm -f registry_2.8.2_linux_amd64.tar.gz", shell=True)
  subprocess.run("chmod +x /usr/local/bin/registry", shell=True)

# 3. Kaniko İndir ve Kur (Docker Imajından Kopyalayarak)
if not os.path.exists("/usr/local/bin/kaniko"):
  print("[INFO] Kaniko executor binary Docker imajından kopyalanıyor...")
  subprocess.run("docker pull gcr.io/kaniko-project/executor:latest", shell=True)
  subprocess.run("docker create --name kaniko-temp gcr.io/kaniko-project/executor:latest", shell=True)
  subprocess.run("docker cp kaniko-temp:/kaniko/executor /usr/local/bin/kaniko", shell=True)
  subprocess.run("docker rm kaniko-temp", shell=True)
  subprocess.run("chmod +x /usr/local/bin/kaniko", shell=True)

# 4. pigz İndir (paralel gzip sıkıştırma için)
subprocess.run("apt-get update && apt-get install -y pigz", shell=True)

print("✅ Kurulumlar tamamlandi.")

# 5. Registry'yi Arka Planda Başlat
print("==========================================")
print("🚀 Yerel Registry localhost:5000 Portunda Baslatiliyor...")
print("==========================================")
os.makedirs("/etc/docker/registry", exist_ok=True)
with open("/etc/docker/registry/config.yml", "w") as cfg_f:
  cfg_f.write("""version: 0.1\nlog:\n  fields:\n    service: registry\nstorage:\n  cache:\n    blobdescriptor: inmemory\n  filesystem:\n    rootdirectory: /var/lib/registry\nhttp:\n  addr: :5000\n  headers:\n    X-Content-Type-Options: [nosniff]\n""")
subprocess.run("pkill -f 'registry serve' || true", shell=True)
# Arka planda registry serve baslat
subprocess.Popen(["/usr/local/bin/registry", "serve", "/etc/docker/registry/config.yml"], stdout=open("registry.log", "w"), stderr=subprocess.STDOUT)

# Registry'nin hazir olmasini bekle
registry_ok = False
for _ in range(15):
  try:
    urllib.request.urlopen("http://localhost:5000/v2/")
    print("✅ Registry başarıyla başlatıldı ve yanıt veriyor!")
    registry_ok = True
    break
  except Exception:
    time.sleep(1)
if not registry_ok:
  print("❌ Hata: Yerel registry başlatılamadı!")
  if os.path.exists("registry.log"):
    with open("registry.log", "r") as log_f:
      print(log_f.read())
  sys.exit(1)

# 6. Insa Betigini Tetikle
print("==========================================")
print("🚀 Docker Imajlarının Insa Edilmesi Baslatiliyor...")
print("==========================================")

os.chdir(repo_dir)

# Son demleri git pull ile cekelim
subprocess.run("git pull origin main || true", shell=True)
subprocess.run("chmod +x colab_docker/build_all.sh", shell=True)

# build_all.sh betigini calistir ve anlik ciktiyi al
process = subprocess.Popen(["bash", "colab_docker/build_all.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
while True:
  output = process.stdout.readline()
  if output == '' and process.poll() is not None:
    break
  if output:
    print(output.strip())

rc = process.poll()
if rc == 0:
  print("✅ Tüm imaj inşaları ve Google Drive yedeklemeleri başarıyla tamamlandı.")
else:
  print(f"❌ İnşa sırasında hata oluştu. Çıkış kodu: {rc}")
  sys.exit(rc)
